# Generate Standardized Lag Feature File `lag.xlsx`

本 notebook 用于从 `merged.xlsx` 构造标准化滞后特征文件 `lag.xlsx`。

核心规则：

1. 不修改原始 `merged.xlsx`；
2. 先对参与滞后构造的原始变量进行 Z-score 标准化；
3. 再基于标准化变量构造 `lag1` 至 `lag12`；
4. 不生成也不保存任何输入变量的 `lag0`；
5. 保留当前时刻 `NTU` 作为模型训练目标变量；
6. 构造 `OP_DATE`，其中 01:00、03:00、05:00 归属于前一天运行日；
7. 将 `R/W PUMP DUTY` 和 `T/W PUMP DUTY` 转换为泵运行台数后再参与滞后构造。

最终输出：

```text
数据文件：与 merged.xlsx 同目录的 lag.xlsx，例如 codes/data/lag.xlsx
备份文件：outputs/problem1/lag.xlsx
标准化参数：outputs/problem1/lag_standardization_info.xlsx
缺失率检查：outputs/problem1/lag_missing_summary.csv
特征名列表：outputs/problem1/lag_feature_columns.txt
```


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# =========================
# 0. Basic settings
# =========================

# This notebook may be run from:
# 1) project root:        2026-Asia-Pacific-cup/
# 2) codes folder:        2026-Asia-Pacific-cup/codes/
# 3) another notebook cwd
#
# Therefore, search several common parent directories automatically.

CWD = Path.cwd().resolve()

search_roots = [CWD] + list(CWD.parents[:5])

candidate_paths = []
for root in search_roots:
    candidate_paths.extend([
        root / "data" / "merged.xlsx",
        root / "codes" / "data" / "merged.xlsx",
        root / "2026-Asia-Pacific-cup" / "codes" / "data" / "merged.xlsx",
    ])

# Remove duplicate paths while preserving order
seen = set()
possible_paths = []
for path in candidate_paths:
    path_resolved = path.resolve()
    if path_resolved not in seen:
        possible_paths.append(path_resolved)
        seen.add(path_resolved)

DATA_PATH = None
for path in possible_paths:
    if path.exists():
        DATA_PATH = path
        break

if DATA_PATH is None:
    print("Current working directory:", CWD)
    print("\nTried paths:")
    for path in possible_paths:
        print(" -", path)
    raise FileNotFoundError(
        "Cannot find merged.xlsx. Please put merged.xlsx under codes/data/ "
        "or run this notebook from the project root / codes folder."
    )

# The data directory is where merged.xlsx is located.
# lag.xlsx will be saved next to merged.xlsx.
DATA_DIR = DATA_PATH.parent

# Base directory is usually codes/.
# outputs/problem1 will be saved under the same base directory as data/.
BASE_DIR = DATA_DIR.parent
OUTPUT_DIR = BASE_DIR / "outputs" / "problem1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Current working directory:", CWD)
print("Using data file:", DATA_PATH)
print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

df = pd.read_excel(DATA_PATH)

print("\nOriginal data shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Current working directory: E:\桌面\亚太杯\2026-Asia-Pasific-cup\temp
Using data file: E:\桌面\亚太杯\2026-Asia-Pasific-cup\data\merged.xlsx
Data directory: E:\桌面\亚太杯\2026-Asia-Pasific-cup\data
Output directory: E:\桌面\亚太杯\2026-Asia-Pasific-cup\outputs\problem1

Original data shape: (5460, 18)
Columns:
['DATE', 'TIME', 'RIVER LEVEL', 'R/W PUMP DUTY', 'R/W FLOW', 'R/W NTU', 'R/W CLR', 'R/W PH', 'FILT. NTU', 'C/W WELL LEVEL', 'PH', 'NTU', 'CLR', 'CL2', 'F/RIDE', 'ALUM', 'T/W PUMP DUTY', 'T/W FLOW']


## 1. Construct `DATETIME` and `OP_DATE`

`DATETIME` 用于保证时序排序正确。

`OP_DATE` 是水厂运行日：

```text
07:00, 09:00, ..., 23:00, next day 01:00, 03:00, 05:00
```

因此：

```text
如果时间 < 07:00，则 OP_DATE = 当前日期 - 1 天
如果时间 >= 07:00，则 OP_DATE = 当前日期
```


In [2]:
# =========================
# 1. Construct DATETIME and OP_DATE
# =========================

required_time_cols = ["DATE", "TIME"]
missing_time_cols = [col for col in required_time_cols if col not in df.columns]
if missing_time_cols:
    raise ValueError(f"Missing required time columns: {missing_time_cols}")

# Process DATE
df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")

# Process TIME robustly
def normalize_time_to_string(x):
    if pd.isna(x):
        return np.nan
    
    # datetime.time or Timestamp-like object
    if hasattr(x, "hour") and hasattr(x, "minute"):
        return f"{x.hour:02d}:{x.minute:02d}:00"
    
    s = str(x).strip()
    
    # Excel time may be read as '1900-01-01 07:00:00'
    if "1900-01-01" in s:
        s = s.replace("1900-01-01", "").strip()
    
    # Handle 700, 0700, 900, 0900, 700.0, 0900.0
    s_no_float = s.replace(".0", "")
    if s_no_float.isdigit():
        s_no_float = s_no_float.zfill(4)
        return f"{s_no_float[:2]}:{s_no_float[2:]}:00"
    
    # Handle 07:00 or 07:00:00
    t = pd.to_datetime(s, errors="coerce")
    if pd.isna(t):
        return np.nan
    
    return t.strftime("%H:%M:%S")


df["TIME_STR"] = df["TIME"].apply(normalize_time_to_string)

# Construct DATETIME
df["DATETIME"] = pd.to_datetime(
    df["DATE"].dt.strftime("%Y-%m-%d") + " " + df["TIME_STR"],
    errors="coerce"
)

# Sort by time
df = df.sort_values("DATETIME").reset_index(drop=True)

# Construct OP_DATE
if df["DATETIME"].isna().any():
    print("Warning: DATETIME has missing values. Missing count:", df["DATETIME"].isna().sum())

df["OP_DATE"] = df["DATETIME"].dt.date
early_morning_mask = df["DATETIME"].dt.hour < 7

df.loc[early_morning_mask, "OP_DATE"] = (
    df.loc[early_morning_mask, "DATETIME"] - pd.Timedelta(days=1)
).dt.date

print("Data shape after datetime processing:", df.shape)
print("DATETIME missing:", df["DATETIME"].isna().sum())
print("Time range:", df["DATETIME"].min(), "to", df["DATETIME"].max())
print("OP_DATE range:", df["OP_DATE"].min(), "to", df["OP_DATE"].max())

df[["DATE", "TIME", "TIME_STR", "DATETIME", "OP_DATE"]].head(15)

Data shape after datetime processing: (5460, 21)
DATETIME missing: 0
Time range: 2025-01-01 07:00:00 to 2026-04-01 05:00:00
OP_DATE range: 2025-01-01 to 2026-03-31


,DATE,TIME,TIME_STR,DATETIME,OP_DATE
0,2025-01-01,07:00:00,07:00:00,2025-01-01 07:00:00,2025-01-01
1,2025-01-01,09:00:00,09:00:00,2025-01-01 09:00:00,2025-01-01
2,2025-01-01,11:00:00,11:00:00,2025-01-01 11:00:00,2025-01-01
3,2025-01-01,13:00:00,13:00:00,2025-01-01 13:00:00,2025-01-01
4,2025-01-01,15:00:00,15:00:00,2025-01-01 15:00:00,2025-01-01
5,2025-01-01,17:00:00,17:00:00,2025-01-01 17:00:00,2025-01-01
6,2025-01-01,19:00:00,19:00:00,2025-01-01 19:00:00,2025-01-01
7,2025-01-01,21:00:00,21:00:00,2025-01-01 21:00:00,2025-01-01
8,2025-01-01,23:00:00,23:00:00,2025-01-01 23:00:00,2025-01-01
9,2025-01-02,01:00:00,01:00:00,2025-01-02 01:00:00,2025-01-01


## 2. Convert `PUMP DUTY` to `PUMP COUNT`

`R/W PUMP DUTY` 和 `T/W PUMP DUTY` 不是普通连续数值。

例如：

```text
1     -> 1 台泵
2     -> 1 台泵
3     -> 1 台泵
1+4   -> 2 台泵
2+3   -> 2 台泵
```

因此需要先转换为：

```text
R/W PUMP COUNT
T/W PUMP COUNT
```


In [3]:
# =========================
# 2. Convert PUMP DUTY to PUMP COUNT
# =========================

def pump_duty_to_count(x):
    if pd.isna(x):
        return np.nan
    
    s = str(x).strip()
    if s == "":
        return np.nan
    
    # Remove common numeric suffix from Excel, e.g. 1.0 -> 1
    if s.endswith(".0"):
        s = s[:-2]
    
    # Combined pump duty, e.g. 1+4, 2+3
    if "+" in s:
        parts = [p.strip() for p in s.split("+") if p.strip() != ""]
        return len(parts) if len(parts) > 0 else np.nan
    
    # Single pump number
    if s.isdigit():
        return 1
    
    return np.nan

pump_source_cols = ["R/W PUMP DUTY", "T/W PUMP DUTY"]
for source_col in pump_source_cols:
    if source_col in df.columns:
        count_col = source_col.replace("DUTY", "COUNT")
        df[count_col] = df[source_col].apply(pump_duty_to_count)
        print(f"Created {count_col} from {source_col}")
        print(df[[source_col, count_col]].drop_duplicates().head(20))
    else:
        print(f"Warning: {source_col} not found. Skipped.")

Created R/W PUMP COUNT from R/W PUMP DUTY
      R/W PUMP DUTY  R/W PUMP COUNT
0               1.0             1.0
1272            NaN             NaN
2949            2.0             1.0
Created T/W PUMP COUNT from T/W PUMP DUTY
      T/W PUMP DUTY  T/W PUMP COUNT
0               2.0             1.0
30              3.0             1.0
1272            NaN             NaN
3073            1.0             1.0


## 3. Define variables for lag construction

这里生成的是 **标准化后的 lag 特征**。

注意：

- `NTU` 是当前时刻目标变量，保留为 `target_NTU`；
- 输入变量只保留 `lag1` 至 `lag12`；
- 不生成 `lag0`；
- `F/RIDE` 虽然缺失率较高，但仍保留为辅助指标，后续建模用中位数插补，不应直接 `dropna`。


In [6]:
# =========================
# 3. Define lag base features
# =========================

TARGET_COL = "NTU"
MAX_LAG = 12
SAMPLING_INTERVAL_HOURS = 2

# Candidate variables for lag construction.
# These are external water-quality / process variables, not current NTU lag0 features.
lag_base_features = [
    "R/W NTU",
    "FILT. NTU",
    "R/W FLOW",
    "T/W FLOW",
    "C/W WELL LEVEL",
    "ALUM",
    "R/W PH",
    "PH",
    "CLR",
    "CL2",
    "RIVER LEVEL",
    "R/W CLR",
    "F/RIDE",
    "R/W PUMP COUNT",
    "T/W PUMP COUNT",
]

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column {TARGET_COL} not found.")

existing_lag_base_features = [col for col in lag_base_features if col in df.columns]
missing_lag_base_features = [col for col in lag_base_features if col not in df.columns]

print("Existing lag base features:")
print(existing_lag_base_features)

print("Missing lag base features, skipped:")
print(missing_lag_base_features)

# Convert target to numeric but do not standardize target here.
# The lag file keeps current NTU as the prediction target.
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")

Existing lag base features:
['R/W NTU', 'FILT. NTU', 'R/W FLOW', 'T/W FLOW', 'C/W WELL LEVEL', 'ALUM', 'R/W PH', 'PH', 'CLR', 'CL2', 'RIVER LEVEL', 'R/W CLR', 'F/RIDE', 'R/W PUMP COUNT', 'T/W PUMP COUNT']
Missing lag base features, skipped:
[]


## 4. Standardize variables before lagging

标准化逻辑：

```text
X_z = (X - mean(X)) / std(X)
```

然后再做：

```text
X_z_lag1 = X_z shifted by 1 row
X_z_lag2 = X_z shifted by 2 rows
...
X_z_lag12 = X_z shifted by 12 rows
```

这样得到的是“标准化后的滞后特征值”，不是“标准化后的相关系数”。


In [7]:
# =========================
# 4. Standardize variables before lag construction
# =========================

standardization_records = []

for col in existing_lag_base_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    
    mean_value = df[col].mean(skipna=True)
    std_value = df[col].std(skipna=True)
    missing_count = df[col].isna().sum()
    missing_rate = missing_count / len(df) if len(df) > 0 else np.nan
    
    z_col = f"{col}_z"
    
    if pd.isna(std_value) or std_value == 0:
        df[z_col] = np.nan
        status = "std_missing_or_zero"
    else:
        df[z_col] = (df[col] - mean_value) / std_value
        status = "standardized"
    
    standardization_records.append({
        "feature": col,
        "standardized_feature": z_col,
        "mean": mean_value,
        "std": std_value,
        "missing_count": missing_count,
        "missing_rate": missing_rate,
        "status": status,
    })

standardization_info_df = pd.DataFrame(standardization_records)

standardization_info_path = OUTPUT_DIR / "lag_standardization_info.xlsx"
standardization_info_df.to_excel(standardization_info_path, index=False)

print("Saved standardization info:", standardization_info_path)
standardization_info_df

Saved standardization info: E:\桌面\亚太杯\2026-Asia-Pasific-cup\outputs\problem1\lag_standardization_info.xlsx


,feature,standardized_feature,mean,std,missing_count,missing_rate,status
0,R/W NTU,R/W NTU_z,38.055128,40.809743,0,0.000000,standardized
1,FILT. NTU,FILT. NTU_z,0.190306,0.578003,0,0.000000,standardized
2,R/W FLOW,R/W FLOW_z,49.341824,3.203787,0,0.000000,standardized
3,T/W FLOW,T/W FLOW_z,45.351066,2.526913,0,0.000000,standardized
4,C/W WELL LEVEL,C/W WELL LEVEL_z,3.710211,0.199635,0,0.000000,standardized
5,ALUM,ALUM_z,0.055314,0.005176,1644,0.301099,standardized
6,R/W PH,R/W PH_z,7.005031,0.022219,1644,0.301099,standardized
7,PH,PH_z,7.293737,0.039059,1644,0.301099,standardized
8,CLR,CLR_z,4.976253,0.329645,0,0.000000,standardized
9,CL2,CL2_z,1.501388,0.259919,1707,0.312637,standardized


## 5. Generate `lag.xlsx`

输出文件保留以下基本列：

```text
DATETIME
OP_DATE
target_NTU
```

然后加入所有标准化滞后特征：

```text
feature_z_lag1
feature_z_lag2
...
feature_z_lag12
```

不会生成：

```text
feature_z_lag0
```


In [8]:
# =========================
# 5. Generate standardized lag features without lag0
# =========================

lag_df = df[["DATETIME", "OP_DATE", TARGET_COL]].copy()
lag_df = lag_df.rename(columns={TARGET_COL: "target_NTU"})

lag_feature_columns = []

for col in existing_lag_base_features:
    z_col = f"{col}_z"
    
    for lag in range(1, MAX_LAG + 1):
        lag_col = f"{col}_z_lag{lag}"
        lag_df[lag_col] = df[z_col].shift(lag)
        lag_feature_columns.append(lag_col)

# Add useful row-level information
lag_df["row_has_target"] = lag_df["target_NTU"].notna()
lag_df["available_lag_feature_count"] = lag_df[lag_feature_columns].notna().sum(axis=1)
lag_df["missing_lag_feature_count"] = lag_df[lag_feature_columns].isna().sum(axis=1)

print("Lag dataframe shape:", lag_df.shape)
print("Number of lag features:", len(lag_feature_columns))
print("First 10 lag feature columns:")
print(lag_feature_columns[:10])

lag_df.head(15)

Lag dataframe shape: (5460, 186)
Number of lag features: 180
First 10 lag feature columns:
['R/W NTU_z_lag1', 'R/W NTU_z_lag2', 'R/W NTU_z_lag3', 'R/W NTU_z_lag4', 'R/W NTU_z_lag5', 'R/W NTU_z_lag6', 'R/W NTU_z_lag7', 'R/W NTU_z_lag8', 'R/W NTU_z_lag9', 'R/W NTU_z_lag10']


C:\Users\HP\AppData\Local\Temp\ipykernel_30316\2536271806.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  lag_df[lag_col] = df[z_col].shift(lag)
C:\Users\HP\AppData\Local\Temp\ipykernel_30316\2536271806.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  lag_df[lag_col] = df[z_col].shift(lag)
C:\Users\HP\AppData\Local\Temp\ipykernel_30316\2536271806.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all c

,DATETIME,OP_DATE,target_NTU,R/W NTU_z_lag1,R/W NTU_z_lag2,R/W NTU_z_lag3,R/W NTU_z_lag4,R/W NTU_z_lag5,R/W NTU_z_lag6,R/W NTU_z_lag7,...,T/W PUMP COUNT_z_lag6,T/W PUMP COUNT_z_lag7,T/W PUMP COUNT_z_lag8,T/W PUMP COUNT_z_lag9,T/W PUMP COUNT_z_lag10,T/W PUMP COUNT_z_lag11,T/W PUMP COUNT_z_lag12,row_has_target,available_lag_feature_count,missing_lag_feature_count
0,2025-01-01 07:00:00,2025-01-01,0.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,0,180
1,2025-01-01 09:00:00,2025-01-01,0.12,2.743092,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,12,168
2,2025-01-01 11:00:00,2025-01-01,0.12,2.007973,2.743092,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,24,156
3,2025-01-01 13:00:00,2025-01-01,0.11,1.076823,2.007973,2.743092,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,36,144
4,2025-01-01 15:00:00,2025-01-01,0.11,0.586744,1.076823,2.007973,2.743092,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,48,132
5,2025-01-01 17:00:00,2025-01-01,0.11,0.439720,0.586744,1.076823,2.007973,2.743092,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,60,120
6,2025-01-01 19:00:00,2025-01-01,0.11,0.905295,0.439720,0.586744,1.076823,2.007973,2.743092,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,72,108
7,2025-01-01 21:00:00,2025-01-01,0.11,1.566902,0.905295,0.439720,0.586744,1.076823,2.007973,2.743092,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,84,96
8,2025-01-01 23:00:00,2025-01-01,0.12,1.493390,1.566902,0.905295,0.439720,0.586744,1.076823,2.007973,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,96,84
9,2025-01-02 01:00:00,2025-01-01,0.12,1.517894,1.493390,1.566902,0.905295,0.439720,0.586744,1.076823,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,108,72


In [9]:
# =========================
# 6. Save lag.xlsx and related check files
# =========================

# Main file for subsequent modeling.
# Save it next to merged.xlsx, e.g. codes/data/lag.xlsx.
main_lag_path = DATA_DIR / "lag.xlsx"
main_lag_path.parent.mkdir(parents=True, exist_ok=True)
lag_df.to_excel(main_lag_path, index=False)

# Backup output file
backup_lag_path = OUTPUT_DIR / "lag.xlsx"
lag_df.to_excel(backup_lag_path, index=False)

# Missing summary
missing_summary = pd.DataFrame({
    "column": lag_df.columns,
    "missing_count": lag_df.isna().sum().values,
    "missing_rate": lag_df.isna().mean().values,
})
missing_summary_path = OUTPUT_DIR / "lag_missing_summary.csv"
missing_summary.to_csv(missing_summary_path, index=False, encoding="utf-8-sig")

# Feature column list
feature_list_path = OUTPUT_DIR / "lag_feature_columns.txt"
with open(feature_list_path, "w", encoding="utf-8") as f:
    for col in lag_feature_columns:
        f.write(col + "\n")

print("Saved main lag file:", main_lag_path)
print("Saved backup lag file:", backup_lag_path)
print("Saved missing summary:", missing_summary_path)
print("Saved feature list:", feature_list_path)

print("\nFinal lag_df shape:", lag_df.shape)
print("Target missing count:", lag_df["target_NTU"].isna().sum())
print("Lag feature columns:", len(lag_feature_columns))

missing_summary.head(20)

Saved main lag file: E:\桌面\亚太杯\2026-Asia-Pasific-cup\data\lag.xlsx
Saved backup lag file: E:\桌面\亚太杯\2026-Asia-Pasific-cup\outputs\problem1\lag.xlsx
Saved missing summary: E:\桌面\亚太杯\2026-Asia-Pasific-cup\outputs\problem1\lag_missing_summary.csv
Saved feature list: E:\桌面\亚太杯\2026-Asia-Pasific-cup\outputs\problem1\lag_feature_columns.txt

Final lag_df shape: (5460, 186)
Target missing count: 336
Lag feature columns: 180


,column,missing_count,missing_rate
0,DATETIME,0,0.000000
1,OP_DATE,0,0.000000
2,target_NTU,336,0.061538
3,R/W NTU_z_lag1,1,0.000183
4,R/W NTU_z_lag2,2,0.000366
5,R/W NTU_z_lag3,3,0.000549
6,R/W NTU_z_lag4,4,0.000733
7,R/W NTU_z_lag5,5,0.000916
8,R/W NTU_z_lag6,6,0.001099
9,R/W NTU_z_lag7,7,0.001282


## 6. Check whether `lag0` was excluded

这一步用于确认输出文件中没有任何输入变量的 `lag0` 特征。


In [10]:
# =========================
# 7. Validation checks
# =========================

lag0_columns = [col for col in lag_df.columns if "lag0" in col]

print("lag0 columns found:", lag0_columns)

if len(lag0_columns) > 0:
    raise ValueError("lag0 columns exist in lag_df. Please check lag generation logic.")
else:
    print("Validation passed: no lag0 feature columns in lag.xlsx.")

# Check selected target operating dates
target_dates = pd.to_datetime([
    "2026-02-01",
    "2026-02-10",
    "2026-02-20",
]).date

target_3days_lag_df = lag_df[lag_df["OP_DATE"].isin(target_dates)].copy()

target_3days_path = OUTPUT_DIR / "target_3days_from_lag_file_check.csv"
target_3days_lag_df.to_csv(target_3days_path, index=False, encoding="utf-8-sig")

print("Saved target 3-days check file:", target_3days_path)
print("Rows per target OP_DATE:")
if len(target_3days_lag_df) > 0:
    print(target_3days_lag_df["OP_DATE"].value_counts().sort_index())
else:
    print("No rows found for target OP_DATE. Check whether these dates exist in merged.xlsx.")

target_3days_lag_df.head(20)

lag0 columns found: []
Validation passed: no lag0 feature columns in lag.xlsx.
Saved target 3-days check file: E:\桌面\亚太杯\2026-Asia-Pasific-cup\outputs\problem1\target_3days_from_lag_file_check.csv
Rows per target OP_DATE:
OP_DATE
2026-02-01    12
2026-02-10    12
2026-02-20    12
Name: count, dtype: int64


,DATETIME,OP_DATE,target_NTU,R/W NTU_z_lag1,R/W NTU_z_lag2,R/W NTU_z_lag3,R/W NTU_z_lag4,R/W NTU_z_lag5,R/W NTU_z_lag6,R/W NTU_z_lag7,...,T/W PUMP COUNT_z_lag6,T/W PUMP COUNT_z_lag7,T/W PUMP COUNT_z_lag8,T/W PUMP COUNT_z_lag9,T/W PUMP COUNT_z_lag10,T/W PUMP COUNT_z_lag11,T/W PUMP COUNT_z_lag12,row_has_target,available_lag_feature_count,missing_lag_feature_count
4752,2026-02-01 07:00:00,2026-02-01,NaN,-0.638454,-0.638454,-0.613950,-0.589446,-0.589446,-0.589446,-0.417918,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,156,24
4753,2026-02-01 09:00:00,2026-02-01,NaN,-0.638454,-0.638454,-0.638454,-0.613950,-0.589446,-0.589446,-0.589446,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,156,24
4754,2026-02-01 11:00:00,2026-02-01,NaN,-0.368910,-0.638454,-0.638454,-0.638454,-0.613950,-0.589446,-0.589446,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,156,24
4755,2026-02-01 13:00:00,2026-02-01,NaN,-0.344406,-0.368910,-0.638454,-0.638454,-0.638454,-0.613950,-0.589446,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,156,24
4756,2026-02-01 15:00:00,2026-02-01,NaN,-0.344406,-0.344406,-0.368910,-0.638454,-0.638454,-0.638454,-0.613950,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,156,24
4757,2026-02-01 17:00:00,2026-02-01,NaN,0.292697,-0.344406,-0.344406,-0.368910,-0.638454,-0.638454,-0.638454,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,156,24
4758,2026-02-01 19:00:00,2026-02-01,NaN,-0.123871,0.292697,-0.344406,-0.344406,-0.368910,-0.638454,-0.638454,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,156,24
4759,2026-02-01 21:00:00,2026-02-01,NaN,-0.221886,-0.123871,0.292697,-0.344406,-0.344406,-0.368910,-0.638454,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,156,24
4760,2026-02-01 23:00:00,2026-02-01,NaN,-0.221886,-0.221886,-0.123871,0.292697,-0.344406,-0.344406,-0.368910,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,156,24
4761,2026-02-02 01:00:00,2026-02-01,NaN,-0.246390,-0.221886,-0.221886,-0.123871,0.292697,-0.344406,-0.344406,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,156,24


## 7. Notes for modeling

这个 `lag.xlsx` 适合后续做：

```text
target_NTU_t = f(X_z_lag1, X_z_lag2, ..., X_z_lag12)
```

建模时注意：

1. 训练集只删除 `target_NTU` 缺失的行；
2. 输入特征缺失值不要直接删除，建议用中位数插补；
3. `F/RIDE` 缺失率较高，但可以保留为辅助特征；
4. RF / XGBoost 不强制需要标准化，但这里的标准化有利于文件结构统一和论文说明；
5. 当前文件不含 `lag0` 输入特征，符合“只记录滞后项，不记录当前输入项”的要求。
